In [ ]:
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL
import pandas as pd
import numpy as np
from sklearn.ensemble import IsolationForest
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [ ]:
# Connect to Snowflake
engine = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

In [ ]:
# hospital_group aggregation — MNR FFS, 2025 only (202501-202512)
df = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count_2025
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt_2025
        , count(distinct case when a.persistentfulladr_cases = 1 then a.case_id end) as persistent_adr_cnt
        , count(distinct case when a.p2p_full_ovtn = 1 then a.case_id end) as p2p_ovrtn_cnt
        , count(distinct case when a.appeal_ovrtn_ind = 1 then a.case_id end) as appeal_ovrtn_cnt
        , count(distinct case when a.mcr_ovtrn_ind = 1 then a.case_id end) as mcr_ovrtn_cnt
        , count(distinct case when a.icm_md_reviewed_ind = 1 then a.case_id end) as md_reviewed_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month between '202501' and '202512'
        and d.collection is not null
    group by 1,2
    having count(distinct a.case_id) >= 30
""", engine)

print(f"Core (2025): {df.shape}")

In [ ]:
# member_appeal — 202601+ only (separate time range)
df_ma = pd.read_sql("""
    select
        d.collection as hospital_group
        , a.fin_market
        , count(distinct a.case_id) as case_count_2026
        , count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) as initial_adr_cnt_2026
        , count(distinct case when a.member_appeal_ind = 1 then a.case_id end) as member_appeal_cnt
        , count(distinct case when a.member_appeal_ovtn_ind = 1 then a.case_id end) as member_appeal_ovtn_cnt
    from tmp_1m.ec_avtar_25_26_3_od as a
    left join tmp_1y.tin_collection as d
        on substr(a.fa_prov_id, 2, 9) = d.tin
    where fin_brand in ('M&R', 'C&S')
        and (
            (ip_type in ('Medical', 'Surgical', 'Transplant')
             and to_varchar(admit_dt_act, 'MM/dd/yyyy') is not null)
            or ip_type in ('LTAC', 'SNF', 'AIR')
        )
        and fin_brand = 'M&R'
        and loc_flag = 1
        and (
            (global_cap = 'NA' and sgr_source_name = 'COSMOS'
             and fin_product_level_3 != 'INSTITUTIONAL' and tfm_include_flag = 1)
            or (sgr_source_name = 'NICE' and nce_tadm_dec_risk_type in ('FFS', 'PHYSICIAN'))
        )
        and hce_admit_month >= '202601'
        and d.collection is not null
    group by 1,2
    having count(distinct case when a.initialfulladr_cases = 1 then a.case_id end) >= 10
""", engine)

engine.dispose()
print(f"Member appeal (202601+): {df_ma.shape}")

In [ ]:
# merge and compute rates
df = (
    df
    .merge(df_ma, on = ["hospital_group", "fin_market"], how = "left")
    .assign(
        initial_adr_rate_2025 = lambda x: x.initial_adr_cnt_2025 / x.case_count_2025,
        persistent_adr_rate = lambda x: x.persistent_adr_cnt / x.case_count_2025,
        p2p_overturn_rate = lambda x: x.p2p_ovrtn_cnt / x.initial_adr_cnt_2025.replace(0, np.nan),
        appeal_overturn_rate = lambda x: x.appeal_ovrtn_cnt / x.initial_adr_cnt_2025.replace(0, np.nan),
        mcr_overturn_rate = lambda x: x.mcr_ovrtn_cnt / x.initial_adr_cnt_2025.replace(0, np.nan),
        md_review_rate = lambda x: x.md_reviewed_cnt / x.case_count_2025,
        member_appeal_rate = lambda x: x.member_appeal_cnt / x.initial_adr_cnt_2026.replace(0, np.nan),
        initial_adr_rate_2026 = lambda x: x.initial_adr_cnt_2026 / x.case_count_2026,
    )
    .dropna(subset = ["p2p_overturn_rate", "member_appeal_rate"])
)

rate_cols = [
    "initial_adr_rate_2025", "persistent_adr_rate", "p2p_overturn_rate",
    "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate",
    "member_appeal_rate", "initial_adr_rate_2026"
]
count_cols = [
    "case_count_2025", "initial_adr_cnt_2025", "persistent_adr_cnt", "p2p_ovrtn_cnt",
    "appeal_ovrtn_cnt", "mcr_ovrtn_cnt", "md_reviewed_cnt",
    "case_count_2026", "member_appeal_cnt", "initial_adr_cnt_2026"
]
df[rate_cols + count_cols] = df[rate_cols + count_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"Shape: {df.shape[0]}")

In [ ]:
# =============================================================================
# DATA PREP: Eligibility filter + Empirical Bayes shrinkage + log-volume features
# =============================================================================

df_iso = df.copy()
df_iso[count_cols] = df_iso[count_cols].replace([np.inf, -np.inf], np.nan).fillna(0)

df_iso = (
    df_iso
    .query("case_count_2025 >= 30 and initial_adr_cnt_2025 >= 10 and initial_adr_cnt_2026 >= 10")
    .copy()
)

k = 50

gr_initial_adr_25 = df_iso["initial_adr_cnt_2025"].sum() / df_iso["case_count_2025"].sum()
gr_persistent = df_iso["persistent_adr_cnt"].sum() / df_iso["case_count_2025"].sum()
gr_p2p = df_iso["p2p_ovrtn_cnt"].sum() / df_iso["initial_adr_cnt_2025"].sum()
gr_appeal = df_iso["appeal_ovrtn_cnt"].sum() / df_iso["initial_adr_cnt_2025"].sum()
gr_mcr = df_iso["mcr_ovrtn_cnt"].sum() / df_iso["initial_adr_cnt_2025"].sum()
gr_md = df_iso["md_reviewed_cnt"].sum() / df_iso["case_count_2025"].sum()
gr_member_appeal = df_iso["member_appeal_cnt"].sum() / df_iso["initial_adr_cnt_2026"].sum()
gr_initial_adr_26 = df_iso["initial_adr_cnt_2026"].sum() / df_iso["case_count_2026"].sum()

df_iso = df_iso.assign(
    initial_adr_rate_2025 = lambda x: (x.initial_adr_cnt_2025 + k * gr_initial_adr_25) / (x.case_count_2025 + k),
    persistent_adr_rate = lambda x: (x.persistent_adr_cnt + k * gr_persistent) / (x.case_count_2025 + k),
    p2p_overturn_rate = lambda x: (x.p2p_ovrtn_cnt + k * gr_p2p) / (x.initial_adr_cnt_2025 + k),
    appeal_overturn_rate = lambda x: (x.appeal_ovrtn_cnt + k * gr_appeal) / (x.initial_adr_cnt_2025 + k),
    mcr_overturn_rate = lambda x: (x.mcr_ovrtn_cnt + k * gr_mcr) / (x.initial_adr_cnt_2025 + k),
    md_review_rate = lambda x: (x.md_reviewed_cnt + k * gr_md) / (x.case_count_2025 + k),
    member_appeal_rate = lambda x: (x.member_appeal_cnt + k * gr_member_appeal) / (x.initial_adr_cnt_2026 + k),
    initial_adr_rate_2026 = lambda x: (x.initial_adr_cnt_2026 + k * gr_initial_adr_26) / (x.case_count_2026 + k),
)

df_iso = df_iso.assign(
    log_case_count_2025 = lambda x: np.log1p(x.case_count_2025),
    log_initial_adr_cnt_2025 = lambda x: np.log1p(x.initial_adr_cnt_2025),
    log_initial_adr_cnt_2026 = lambda x: np.log1p(x.initial_adr_cnt_2026),
)

iso_kpi = [
    "log_case_count_2025", "log_initial_adr_cnt_2025", "log_initial_adr_cnt_2026",
    "initial_adr_rate_2025", "persistent_adr_rate", "p2p_overturn_rate",
    "appeal_overturn_rate", "mcr_overturn_rate", "md_review_rate",
    "member_appeal_rate", "initial_adr_rate_2026"
]

df_iso[iso_kpi] = df_iso[iso_kpi].replace([np.inf, -np.inf], np.nan).fillna(0)

print(f"IF-ready: {df_iso.shape[0]} hospitals, {len(iso_kpi)} features")

In [ ]:
# =============================================================================
# ISOLATION FOREST: anomaly detection
# =============================================================================
scaler_iso = StandardScaler()
X_iso = scaler_iso.fit_transform(df_iso[iso_kpi])

iso_model = IsolationForest(contamination = 0.05, n_estimators = 300, random_state = 42)
iso_model.fit(X_iso)

df_iso = df_iso.assign(
    anomaly_score = iso_model.decision_function(X_iso),
    anomaly_flag = iso_model.predict(X_iso) == -1,
)
df_iso["anomaly_rank"] = df_iso["anomaly_score"].rank(method = "dense", ascending = True).astype(int)

print(f"Flagged: {df_iso['anomaly_flag'].sum()} / {df_iso.shape[0]} hospitals")

In [ ]:
# =============================================================================
# SHAP: Explain WHY Isolation Forest flagged each hospital
# =============================================================================
import shap

explainer = shap.TreeExplainer(iso_model)
shap_values = explainer.shap_values(X_iso)

print(f"SHAP values shape: {shap_values.shape}")
print(f"Features: {iso_kpi}")

In [ ]:
# =============================================================================
# SHAP SUMMARY PLOT: Global feature importance
# =============================================================================
shap.summary_plot(shap_values, X_iso, feature_names = iso_kpi, show = False)
plt.title("SHAP Summary: Which features drive anomaly flags?")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SHAP BAR PLOT: Mean absolute SHAP value per feature
# =============================================================================
shap.summary_plot(shap_values, X_iso, feature_names = iso_kpi, plot_type = "bar", show = False)
plt.title("SHAP: Mean Feature Importance for Isolation Forest")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# SHAP FOR FLAGGED HOSPITALS: Per-hospital driver identification
# =============================================================================

flagged_idx = df_iso[df_iso["anomaly_flag"]].index
flagged_positions = [df_iso.index.get_loc(idx) for idx in flagged_idx]

shap_flagged = pd.DataFrame(
    shap_values[flagged_positions],
    columns = iso_kpi,
    index = flagged_idx
)

shap_flagged["hospital_group"] = df_iso.loc[flagged_idx, "hospital_group"].values
shap_flagged["fin_market"] = df_iso.loc[flagged_idx, "fin_market"].values
shap_flagged["anomaly_rank"] = df_iso.loc[flagged_idx, "anomaly_rank"].values

shap_flagged["shap_driver"] = shap_flagged[iso_kpi].idxmin(axis = 1)
shap_flagged["shap_driver_value"] = shap_flagged[iso_kpi].min(axis = 1)

print(f"Flagged: {len(shap_flagged)} hospitals")
print("="*60)
shap_flagged[["hospital_group", "fin_market", "anomaly_rank", "shap_driver", "shap_driver_value"]].sort_values("anomaly_rank")

In [ ]:
# =============================================================================
# SHAP WATERFALL: Top 5 most anomalous hospitals
# =============================================================================

top5_idx = (
    df_iso[df_iso["anomaly_flag"] & df_iso["hospital_group"].notna()]
    .nsmallest(5, "anomaly_score")
    .index
)

base_val = float(explainer.expected_value) if np.ndim(explainer.expected_value) == 0 else float(explainer.expected_value[0])

for idx in top5_idx:
    pos = df_iso.index.get_loc(idx)
    hospital_name = df_iso.loc[idx, "hospital_group"]
    rank = int(df_iso.loc[idx, "anomaly_rank"])

    print(f"\n{'='*60}")
    print(f"{hospital_name} | Rank #{rank}")
    print(f"{'='*60}")

    explanation = shap.Explanation(
        values = shap_values[pos],
        base_values = base_val,
        feature_names = iso_kpi
    )
    shap.waterfall_plot(explanation, show = False)
    plt.title(f"{hospital_name[:35]} | Rank #{rank}")
    plt.tight_layout()
    plt.show()

In [ ]:
# =============================================================================
# SHAP FORCE PLOT: Interactive view for #1 most anomalous hospital
# =============================================================================

top1_pos = df_iso.index.get_loc(top5_idx[0])
top1_name = df_iso.loc[top5_idx[0], "hospital_group"]

print(f"Force plot for: {top1_name} (Rank #1)")
shap.force_plot(
    base_val,
    shap_values[top1_pos],
    feature_names = iso_kpi,
    matplotlib = True,
    show = False
)
plt.title(f"SHAP Force Plot: {top1_name[:35]}")
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================================
# OUTPUT ASSEMBLY
# =============================================================================

df_iso["driver"] = shap_flagged["shap_driver"].reindex(df_iso.index).fillna("")
df_iso["direction"] = np.where(
    df_iso["anomaly_flag"],
    df_iso.apply(lambda r: "high" if r["driver"] and df_iso.loc[r.name, r["driver"]] > df_iso[r["driver"]].median() else "low", axis=1),
    ""
)
df_iso["reason"] = np.where(df_iso["anomaly_flag"], df_iso["driver"] + " (" + df_iso["direction"] + ")", "")

pop_medians = df_iso[iso_kpi].median()
raw_med_case_count_2025 = df_iso["case_count_2025"].median()
raw_med_initial_adr_2025 = df_iso["initial_adr_cnt_2025"].median()
raw_med_initial_adr_2026 = df_iso["initial_adr_cnt_2026"].median()

df_iso["driver_value"] = np.nan
df_iso["driver_pop_median"] = np.nan
df_iso["driver_dev"] = np.nan

mask = df_iso["anomaly_flag"] & (df_iso["driver"] == "log_case_count_2025")
df_iso.loc[mask, "driver_value"] = df_iso.loc[mask, "case_count_2025"]
df_iso.loc[mask, "driver_pop_median"] = raw_med_case_count_2025
df_iso.loc[mask, "driver_dev"] = (df_iso.loc[mask, "case_count_2025"] - raw_med_case_count_2025) / raw_med_case_count_2025

mask = df_iso["anomaly_flag"] & (df_iso["driver"] == "log_initial_adr_cnt_2025")
df_iso.loc[mask, "driver_value"] = df_iso.loc[mask, "initial_adr_cnt_2025"]
df_iso.loc[mask, "driver_pop_median"] = raw_med_initial_adr_2025
df_iso.loc[mask, "driver_dev"] = (df_iso.loc[mask, "initial_adr_cnt_2025"] - raw_med_initial_adr_2025) / raw_med_initial_adr_2025

mask = df_iso["anomaly_flag"] & (df_iso["driver"] == "log_initial_adr_cnt_2026")
df_iso.loc[mask, "driver_value"] = df_iso.loc[mask, "initial_adr_cnt_2026"]
df_iso.loc[mask, "driver_pop_median"] = raw_med_initial_adr_2026
df_iso.loc[mask, "driver_dev"] = (df_iso.loc[mask, "initial_adr_cnt_2026"] - raw_med_initial_adr_2026) / raw_med_initial_adr_2026

for rate_col in rate_cols:
    mask = df_iso["anomaly_flag"] & (df_iso["driver"] == rate_col)
    if mask.any():
        df_iso.loc[mask, "driver_value"] = df_iso.loc[mask, rate_col]
        df_iso.loc[mask, "driver_pop_median"] = pop_medians[rate_col]
        df_iso.loc[mask, "driver_dev"] = (df_iso.loc[mask, rate_col] - pop_medians[rate_col]) / pop_medians[rate_col]

output_cols = [
    "hospital_group", "fin_market",
    "case_count_2025", "initial_adr_cnt_2025", "case_count_2026", "initial_adr_cnt_2026",
    *iso_kpi,
    "anomaly_score", "anomaly_rank", "anomaly_flag",
    "driver", "direction", "reason",
    "driver_value", "driver_pop_median", "driver_dev"
]

df_iso_output = df_iso[output_cols].sort_values("anomaly_score").reset_index(drop = True)

print(f"Model input: {df_iso.shape[0]} entities | Flagged: {df_iso['anomaly_flag'].sum()}")
df_iso_output

In [ ]:
# =============================================================================
# WRITE FLAGGED HOSPITALS TO SNOWFLAKE
# =============================================================================
from sqlalchemy import create_engine
from snowflake.sqlalchemy import URL

engine_write = create_engine(URL(
    account = "UHG-UHGDWAAS",
    user = "KHANG.NGUYEN@UHC.COM",
    authenticator = "externalbrowser",
    role = "AZU_SDRP_VING_PRD_DEVELOPER_ROLE",
    warehouse = "VING_PRD_MNR_HCE_DATAINFRA_WH",
    database = "VING_PRD_TREND_DB",
    schema = "TMP_1M"
))

flagged = df_iso_output[df_iso_output["anomaly_flag"] == True].copy()
flagged["shap_driver"] = flagged["driver"]

shap_lookup = shap_flagged[["hospital_group", "fin_market", "shap_driver"]].reset_index(drop = True)
flagged = flagged.merge(
    shap_lookup,
    on = ["hospital_group", "fin_market"],
    how = "left",
    suffixes = ("", "_lookup")
)
if "shap_driver_lookup" in flagged.columns:
    flagged["shap_driver"] = flagged["shap_driver_lookup"]
    flagged.drop(columns = ["shap_driver_lookup"], inplace = True)

flagged.columns = [c.upper() for c in flagged.columns]

flagged.to_sql(
    "kn_loc_if_outlier_hospitals_mnr",
    engine_write,
    schema = "TMP_1M",
    if_exists = "replace",
    index = False
)

engine_write.dispose()
print(f"Wrote {len(flagged)} flagged hospitals to TMP_1M.KN_LOC_IF_OUTLIER_HOSPITALS_MNR")

In [ ]:
# =============================================================================
# POST-HOC ANALYSIS: MAD-Z scores (validation/comparison only)
# - MAD z = (x - median) / (MAD * 1.4826)
# - Used as a sanity check against SHAP drivers, NOT as primary explanation
# =============================================================================
from scipy.stats import median_abs_deviation

mad_z_cols = [f"{c}_z" for c in iso_kpi]

for col in iso_kpi:
    med = df_iso[col].median()
    mad_scaled = median_abs_deviation(df_iso[col], scale = 1.0) * 1.4826
    df_iso[f"{col}_z"] = (df_iso[col] - med) / mad_scaled if mad_scaled > 0 else 0

df_iso["mad_z_driver"] = df_iso[mad_z_cols].abs().idxmax(axis = 1).str.removesuffix("_z")
df_iso["mad_z_score"] = df_iso[mad_z_cols].apply(lambda r: r[r.abs().idxmax()], axis = 1)

print(f"MAD z-scores computed for {len(iso_kpi)} features")

In [ ]:
# =============================================================================
# POST-HOC: SHAP vs MAD-Z agreement check
# =============================================================================

flagged_comparison = df_iso[df_iso["anomaly_flag"]][["hospital_group", "fin_market", "anomaly_rank", "driver", "mad_z_driver"]].copy()
flagged_comparison.rename(columns = {"driver": "shap_driver"}, inplace = True)
flagged_comparison["drivers_agree"] = flagged_comparison["shap_driver"] == flagged_comparison["mad_z_driver"]

print(f"SHAP vs MAD-Z agreement: {flagged_comparison['drivers_agree'].sum()} / {len(flagged_comparison)} ({flagged_comparison['drivers_agree'].mean():.0%})")
print("="*60)
flagged_comparison.sort_values("anomaly_rank")

In [ ]:
# =============================================================================
# POST-HOC: Disagreement detail
# =============================================================================
disagree = flagged_comparison[~flagged_comparison["drivers_agree"]].sort_values("anomaly_rank")

print(f"{len(disagree)} hospitals where SHAP and MAD-Z disagree on driver:")
print("="*80)
disagree